# 3. Explore your data (tables)

For every question you named in `config.json`, this prints a simple table: a **number** question (like age) gives count/average/min/max/quartiles; a **matrix** grid gives the breakdown for each row; a **select-all** question gives how many picked each option; anything else gives a count of each answer.

No charts yet (see the README, "What's not done yet"). Tables only, and you never touch pandas. If a quality report exists, low-quality responses are dropped first; set `EXCLUDE_FLAGGED = False` to keep everyone.

In [ ]:
# --- standard setup (works on any OS; uses relative paths only) ---
from pathlib import Path
import json, re
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":          # launched from inside notebooks/
    ROOT = ROOT.parent
DATA_DIR = ROOT / "sample_data"       # put your own Qualtrics CSV here
OUT_DIR  = ROOT / "output"            # everything this tool writes lands here
OUT_DIR.mkdir(exist_ok=True)
print("Repo root :", ROOT)
print("Data dir  :", DATA_DIR)
print("Output dir:", OUT_DIR)

In [ ]:
config = json.loads((OUT_DIR / "config.json").read_text())
clean = pd.read_csv(OUT_DIR / "responses_clean.csv", dtype=str, keep_default_na=False)

EXCLUDE_FLAGGED = True
report_path = OUT_DIR / "quality_report.csv"
if EXCLUDE_FLAGGED and report_path.exists():
    rep = pd.read_csv(report_path)
    id_col = "ResponseId" if "ResponseId" in clean.columns else clean.columns[0]
    drop_ids = set(rep.loc[rep["exclude_recommended"], id_col].astype(str))
    before = len(clean)
    clean = clean[~clean[id_col].astype(str).isin(drop_ids)].reset_index(drop=True)
    print(f"Dropped {before - len(clean)} flagged response(s); {len(clean)} remain.")
else:
    print(f"Using all {len(clean)} responses.")

## Helper that prints the right table for each question type

In [ ]:
cb_path = OUT_DIR / "codebook.csv"
text_by_code = {}
if cb_path.exists():
    _c = pd.read_csv(cb_path)
    text_by_code = dict(zip(_c["code"], _c["question_text"]))

def is_numeric(series):
    vals = [v for v in series if str(v).strip() != ""]
    if not vals:
        return False
    try:
        [float(v) for v in vals]
        return True
    except ValueError:
        return False

def show_question(name, base):
    print("=" * 70)
    print(f"{name}   (question {base})")
    print("-" * 70)
    matrix = config.get("straightline_groups", {})

    if base in matrix:                                   # matrix grid
        for item in matrix[base]:
            if item not in clean.columns:
                continue
            label = str(text_by_code.get(item, item)).split(" - ")[-1]
            counts = clean[item].replace("", pd.NA).dropna().value_counts().sort_index()
            print(f"  {label}")
            print("    " + counts.to_string().replace("\n", "\n    "))
            print()
        return

    cols = [c for c in clean.columns if c == base or c.startswith(base + "_")]
    cols = [c for c in cols if not c.endswith("_TEXT")] or [base]
    col = cols[0] if cols and cols[0] in clean.columns else None
    if col is None:
        print("  (column not found in data)")
        return

    series = clean[col].replace("", pd.NA).dropna()
    if any("," in str(v) for v in series):               # select-all (comma joined)
        tokens = series.str.split(",").explode().str.strip()
        print(tokens.value_counts().to_string())
    elif is_numeric(series):                             # numeric
        print(pd.to_numeric(series).describe().to_string())
    else:                                                # categorical
        print(series.value_counts().to_string())
    print()

## Run it for every named question

In [ ]:
for name, base in config["questions"].items():
    show_question(name, base)